# Advanced Agent Memory: Layering, Forgetting, Conflict Resolution, and Evaluation

这份 notebook 是对 `2-3-ai-agents-for-beginners/13-agent-memory` 里 memory 部分的深入补全。

前面的课程已经让你看到：

- agent 可以把用户偏好写进 memory
- memory 可以被向量化存储和检索

但更深层的系统问题还包括：

- semantic / episodic / procedural memory 如何分层
- memory decay / forgetting 怎么做
- memory conflict resolution 怎么做
- long-term memory quality 怎么评估
- memory write policy 应该怎样设计


## 这一节的技术在做什么

`Memory` 真正难的地方，不是“能不能存”，而是：

- 存什么
- 存到哪一层
- 什么时候忘掉
- 如果新旧记忆冲突怎么办
- 怎样判断当前 memory 系统真的在帮 agent，而不是污染它

这份 notebook 会继续沿用前面 travel assistant 的例子，但把 memory 从“一个向量库”扩展成一个更接近真实系统的分层结构。


## Imports


In [ ]:
import math
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Define layered memory types


In [ ]:
def find_local_bge_snapshot() -> Path:
    snapshot_root = (
        Path("/Users/a1-6/Desktop/AIAgent/models")
        / "models--BAAI--bge-small-en-v1.5"
        / "snapshots"
    )
    snapshots = sorted(
        path for path in snapshot_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    )
    if not snapshots:
        raise FileNotFoundError(f"No valid BGE snapshot found under {snapshot_root}")
    return snapshots[0]


embed_model = SentenceTransformer(str(find_local_bge_snapshot()))


@dataclass
class MemoryRecord:
    memory_id: str
    layer: str
    text: str
    topic_key: str
    confidence: float
    importance: float
    created_at: float
    last_accessed_at: float
    metadata: dict[str, Any] = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None

    def touch(self) -> None:
        self.last_accessed_at = time.time()


class LayeredMemoryStore:
    # 这里显式拆出三层：
    # 1. semantic memory: 稳定偏好 / 用户长期事实
    # 2. episodic memory: 某次任务、某次旅程、某次事件
    # 3. procedural memory: 系统规则、工作流程、行为约束
    def __init__(self):
        self.semantic: list[MemoryRecord] = []
        self.episodic: list[MemoryRecord] = []
        self.procedural: list[MemoryRecord] = []

    def all_records(self) -> list[MemoryRecord]:
        return self.semantic + self.episodic + self.procedural


## 2. Implement a write policy


In [ ]:
store = LayeredMemoryStore()


def classify_memory_layer(text: str) -> tuple[str, str]:
    # 这就是一个最小版 memory write policy。
    # 它回答的是：一段新信息来了，应该写进哪一层？
    lowered = text.lower()

    if any(keyword in lowered for keyword in ["always", "must", "policy", "workflow", "ask before", "rule"]):
        return "procedural", "policy_rule"
    if any(keyword in lowered for keyword in ["this trip", "for this anniversary", "last booking", "during our stay", "on the last trip"]):
        return "episodic", "trip_event"
    return "semantic", "user_preference"


def build_memory_record(memory_id: str, text: str, confidence: float, importance: float) -> MemoryRecord:
    layer, topic_key = classify_memory_layer(text)
    vector = embed_model.encode([text], normalize_embeddings=True)[0]
    now = time.time()
    return MemoryRecord(
        memory_id=memory_id,
        layer=layer,
        text=text,
        topic_key=topic_key,
        confidence=confidence,
        importance=importance,
        created_at=now,
        last_accessed_at=now,
        metadata={},
        embedding=vector,
    )


def write_memory(record: MemoryRecord) -> None:
    # 先做 conflict resolution，再真正落到对应层。
    resolve_conflict(record)
    if record.layer == "semantic":
        store.semantic.append(record)
    elif record.layer == "episodic":
        store.episodic.append(record)
    else:
        store.procedural.append(record)


## 3. Handle conflict resolution


In [ ]:
def resolve_conflict(new_record: MemoryRecord) -> None:
    # 最小版 conflict policy：
    # 如果同一层里已有“高度相似且 topic_key 相同”的记忆，就比较置信度和新旧程度。
    candidate_pool = {
        "semantic": store.semantic,
        "episodic": store.episodic,
        "procedural": store.procedural,
    }[new_record.layer]

    survivors = []
    for old_record in candidate_pool:
        same_topic = old_record.topic_key == new_record.topic_key
        similarity = cosine_similarity(
            [old_record.embedding],
            [new_record.embedding],
        )[0][0]

        if same_topic and similarity > 0.85:
            # 冲突时优先保留“更高 confidence”；如果 confidence 接近，就保留更新的那条。
            replace_old = (
                new_record.confidence > old_record.confidence
                or (
                    abs(new_record.confidence - old_record.confidence) < 0.05
                    and new_record.created_at >= old_record.created_at
                )
            )
            if not replace_old:
                survivors.append(old_record)
        else:
            survivors.append(old_record)

    candidate_pool[:] = survivors


## 4. Add forgetting / decay for episodic memory


In [ ]:
def episodic_decay_score(record: MemoryRecord, now: float) -> float:
    # 真实系统里通常不会对 semantic / procedural 轻易遗忘，
    # 但 episodic memory 很容易无限增长，所以更需要衰减。
    age_hours = (now - record.created_at) / 3600
    recency_hours = (now - record.last_accessed_at) / 3600

    # importance 越高、最近访问越频繁，衰减就越慢。
    raw = math.exp(-0.08 * age_hours) * math.exp(-0.03 * recency_hours) * (0.5 + record.importance)
    return raw


def prune_episodic_memory(threshold: float = 0.25) -> pd.DataFrame:
    now = time.time()
    kept = []
    rows = []
    for record in store.episodic:
        score = episodic_decay_score(record, now)
        rows.append(
            {
                "memory_id": record.memory_id,
                "text": record.text,
                "importance": record.importance,
                "decay_score": score,
                "will_keep": score >= threshold,
            }
        )
        if score >= threshold:
            kept.append(record)

    store.episodic = kept
    return pd.DataFrame(rows)


## 5. Populate the store with travel-agent style memories


In [ ]:
seed_memories = [
    ("m1", "The user prefers vegetarian-friendly restaurants.", 0.95, 0.9),
    ("m2", "The user has a nut allergy and hotel dining recommendations must take that into account.", 0.99, 1.0),
    ("m3", "For this anniversary trip, the user wants romantic destinations and spa experiences.", 0.9, 0.85),
    ("m4", "On the last trip, the user complained that the hotel gym was too small.", 0.7, 0.45),
    ("m5", "Always ask for travel dates before confirming hotel availability.", 1.0, 1.0),
    ("m6", "The user budget is around $700-800 per night for special occasions.", 0.92, 0.8),
    ("m7", "For this trip, the husband has mobility issues and needs accessible accommodations.", 0.97, 0.95),
    ("m8", "The user now says the budget can stretch to $900 if accessibility is excellent.", 0.82, 0.75),
]

for memory_id, text, confidence, importance in seed_memories:
    write_memory(build_memory_record(memory_id, text, confidence, importance))

print("Semantic:", len(store.semantic))
print("Episodic:", len(store.episodic))
print("Procedural:", len(store.procedural))


## 6. Retrieve from layered memory


In [ ]:
def retrieve_memories(query: str, top_k: int = 5) -> pd.DataFrame:
    query_embedding = embed_model.encode([query], normalize_embeddings=True)[0]
    rows = []
    for record in store.all_records():
        similarity = cosine_similarity([query_embedding], [record.embedding])[0][0]
        rows.append(
            {
                "memory_id": record.memory_id,
                "layer": record.layer,
                "topic_key": record.topic_key,
                "similarity": float(similarity),
                "confidence": record.confidence,
                "importance": record.importance,
                "text": record.text,
            }
        )
        record.touch()

    # 最终排序时不只看相似度，还轻微考虑 confidence 和 importance，
    # 这能避免“语义上像，但其实不可靠”的记忆排得过高。
    df = pd.DataFrame(rows)
    df["final_score"] = (
        0.7 * df["similarity"]
        + 0.2 * df["confidence"]
        + 0.1 * df["importance"]
    )
    return df.sort_values("final_score", ascending=False).head(top_k).reset_index(drop=True)


retrieve_memories("Find hotels for an anniversary trip with accessibility needs and careful food handling.")


## 7. Evaluate long-term memory quality


In [ ]:
# 这里做一个最小版 memory quality evaluation：
# 看关键 query 是否能把正确层、正确信息召回到前面。
eval_queries = [
    {
        "query": "What food restrictions should the travel agent remember?",
        "expected_phrase": "nut allergy",
    },
    {
        "query": "What process rule should the agent follow before confirming availability?",
        "expected_phrase": "ask for travel dates",
    },
    {
        "query": "What accessibility needs matter for this trip?",
        "expected_phrase": "accessible accommodations",
    },
]

eval_rows = []
for item in eval_queries:
    retrieved = retrieve_memories(item["query"], top_k=3)
    joined_text = " || ".join(retrieved["text"].tolist()).lower()
    hit = item["expected_phrase"].lower() in joined_text
    eval_rows.append(
        {
            "query": item["query"],
            "expected_phrase": item["expected_phrase"],
            "top3_contains_expected": hit,
        }
    )

pd.DataFrame(eval_rows)


In [ ]:
# 最后看一下 forgetting 之前和之后 episodic memory 的变化。
before_prune = len(store.episodic)
decay_report = prune_episodic_memory(threshold=0.3)
after_prune = len(store.episodic)

print(f"Episodic memory count before prune: {before_prune}")
print(f"Episodic memory count after prune: {after_prune}")
decay_report
